## Multi head attention

In [3]:
import numpy as np
import torch
import torch.nn as nn
import math

In [12]:
class CausalAttention(nn.Module):
    def __init__(self, dim_in, dim_out, context_length, dropout_percentage=0.0, qkv_bias=False):
        super().__init__()

        self.dim_in = dim_in
        self.dim_out = dim_out
        self.context_length = context_length

        self.dropout = nn.Dropout(p=dropout_percentage)
        
        self.weight_query = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.weight_value = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.weight_key = nn.Linear(dim_in, dim_out, bias=qkv_bias)

        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, inputs):
        queries = self.weight_query(inputs)
        keys = self.weight_key(inputs)
        values = self.weight_value(inputs)

        attention_scores = queries @ keys.T 

        scaled_weights = attention_scores / math.sqrt(keys.shape[0])

        causal_attention_scores = attention_scores.masked_fill(self.mask.bool(), -torch.inf)

        causal_scaled_weights = causal_attention_scores / math.sqrt(keys.shape[0])

        causal_attention_weights = torch.softmax(causal_scaled_weights, dim=1)
        
        causal_attention_weights = self.dropout(causal_attention_weights)

        context_vector = causal_attention_weights @ values
        
        return context_vector

## implement the multi head attention

In [13]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, dim_in, dim_out, context_length, number_of_heads,  drop_percentage=0.0, qkv_bias=False):
        super().__init__()

        self.heads = nn.ModuleList([CausalAttention(dim_in, dim_out, context_length, drop_percentage, qkv_bias) for _ in range(number_of_heads)])
    
    def forward(self, x):
        final_context_vector = []
        for head in self.heads:
            context_vector = head(x)
            final_context_vector.append(context_vector)
        return final_context_vector    

### test multi head attention class

In [14]:
# input embeddings
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your 
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]  # step
])

In [17]:
dim_in = inputs.shape[-1]
dim_out = 2
context_length = inputs.shape[0]

multi_head_causal_attention = MultiHeadAttentionWrapper(dim_in= dim_in,dim_out=dim_out, context_length=context_length,number_of_heads=3)

context_vector = multi_head_causal_attention(inputs)
context_vector

[tensor([[-0.1462, -0.5619],
         [-0.2802, -0.7174],
         [-0.3225, -0.7661],
         [-0.3092, -0.6991],
         [-0.2731, -0.6444],
         [-0.2875, -0.6392]], grad_fn=<MmBackward0>),
 tensor([[-0.3925,  0.2792],
         [-0.3422,  0.3828],
         [-0.3222,  0.4083],
         [-0.2794,  0.3856],
         [-0.2253,  0.2956],
         [-0.2330,  0.3431]], grad_fn=<MmBackward0>),
 tensor([[ 0.1529,  0.1104],
         [ 0.1173, -0.0915],
         [ 0.1048, -0.1489],
         [ 0.0894, -0.1685],
         [ 0.0675, -0.1094],
         [ 0.0704, -0.1590]], grad_fn=<MmBackward0>)]